# stagev2 visualization

This notebook reads generated stagev2 outputs. It does not retrain models and does not call APIs.

Before opening the notebook, generate the reports from the `stagev2` project root (the directory containing `run_stagev2.py`, `requirements.txt`, `src/`, and `stage_core/`) with the current system or Conda Python:

```powershell
python -m pip install -r requirements.txt
python .\run_stagev2.py --data-root .\input\raw
```


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

OUTPUT_DIR = Path('../output/stagev2/final_report')
RUN_DIR = Path('../output/stagev2/run_external_accuracy_selection')

ranking_path = OUTPUT_DIR / 'stagev2_model_ranking_by_external_accuracy.csv'
external_path = OUTPUT_DIR / 'stagev2_external_performance_report.csv'
cv_path = OUTPUT_DIR / 'stagev2_cv_summary.csv'
pred_path = OUTPUT_DIR / 'stagev2_test_predictions_all_models.csv'
oof_path = OUTPUT_DIR / 'stagev2_oof_predictions_top10.csv'

for p in [ranking_path, external_path, cv_path, pred_path, oof_path]:
    print(p, 'exists=', p.exists())


## 1. Top 20 models by external accuracy


In [ ]:
external = pd.read_csv(external_path)
top = external.sort_values(['accuracy', 'balanced_accuracy'], ascending=False).head(20).copy()
top_cols = [
    'model_name', 'group', 'feature_block', 'accuracy', 'balanced_accuracy',
    'sensitivity', 'specificity', 'f1', 'roc_auc', 'pr_auc', 'mcc', 'tn', 'fp', 'fn', 'tp'
]
external_metric_names = {
    col: f'external_{col}'
    for col in ['accuracy', 'balanced_accuracy', 'sensitivity', 'specificity', 'f1', 'roc_auc', 'pr_auc', 'mcc', 'tn', 'fp', 'fn', 'tp']
}
display(
    top[[c for c in top_cols if c in top.columns]]
    .rename(columns=external_metric_names)
    .reset_index(drop=True)
)

plt.figure(figsize=(10, 7))
plt.barh(top['model_name'][::-1], top['accuracy'][::-1])
plt.xlabel('External accuracy')
plt.ylabel('Model')
plt.title('Top 20 models by external accuracy')
plt.tight_layout()
plt.show()


## 2. All-model external metrics table


In [ ]:
external_cols = [
    'model_name', 'group', 'feature_block', 'accuracy', 'balanced_accuracy',
    'sensitivity', 'specificity', 'f1', 'roc_auc', 'pr_auc', 'mcc', 'tn', 'fp', 'fn', 'tp'
]
external[external_cols].sort_values(
    ['accuracy', 'balanced_accuracy', 'sensitivity', 'specificity', 'f1', 'roc_auc', 'pr_auc'],
    ascending=False
).rename(columns=external_metric_names).reset_index(drop=True)


## 3. Sensitivity and specificity comparison


In [ ]:
top_metrics = (
    top[['model_name', 'sensitivity', 'specificity']]
    .rename(columns=external_metric_names)
    .set_index('model_name')
)
top_metrics.iloc[::-1].plot(kind='barh', figsize=(10, 7))
plt.xlabel('Metric value')
plt.ylabel('Model')
plt.title('Sensitivity and specificity for top external-accuracy models')
plt.tight_layout()
plt.show()
top_metrics


## 4. CV diagnostic table


In [ ]:
cv = pd.read_csv(cv_path)
cv_cols = [
    'model_name', 'group', 'feature_block', 'cv_accuracy', 'cv_balanced_accuracy',
    'cv_sensitivity', 'cv_specificity', 'cv_f1', 'cv_roc_auc', 'cv_pr_auc', 'cv_mcc',
    'best_cv_score_grid', 'best_params'
]
cv[[c for c in cv_cols if c in cv.columns]].sort_values(['cv_accuracy', 'cv_balanced_accuracy'], ascending=False).reset_index(drop=True)


## 5. Selected model confusion matrix and probability distribution


In [ ]:
pred = pd.read_csv(pred_path)
selected_model = external.sort_values(['accuracy', 'balanced_accuracy'], ascending=False).iloc[0]['model_name']
sel_pred = pred[pred['model_name'] == selected_model].copy()
print('Selected model:', selected_model)
conf = pd.crosstab(sel_pred['y_true'], sel_pred['y_pred'], rownames=['True'], colnames=['Predicted'], dropna=False)
display(conf)
plt.figure(figsize=(7, 4))
plt.hist(sel_pred['p_ad'], bins=20)
plt.xlabel('Predicted AD probability')
plt.ylabel('Number of samples')
plt.title('Selected model probability distribution')
plt.tight_layout()
plt.show()


## 6. early / middle / late subgroup error analysis


In [ ]:
subgroup = (
    sel_pred.groupby('severity_group', dropna=False)
    .agg(n=('sample_id','count'), external_accuracy=('correct','mean'), mean_p_ad=('p_ad','mean'), external_fn=('error_type', lambda x: (x.astype(str).str.startswith('FN')).sum()), external_fp=('error_type', lambda x: (x == 'FP_normal').sum()))
    .reset_index()
)
display(subgroup)
plt.figure(figsize=(8, 4))
plt.bar(subgroup['severity_group'].astype(str), subgroup['external_accuracy'])
plt.xlabel('Severity group')
plt.ylabel('Accuracy')
plt.title('Selected model subgroup accuracy')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()


## 7. Top 10 OOF prediction error table


In [ ]:
oof = pd.read_csv(oof_path)
oof_errors = oof[oof['correct'] == 0].copy()
display(oof_errors[['model_name','sample_id','y_true','y_pred','p_ad','severity_group','mmse','error_type']].sort_values(['model_name','error_type']).head(100))


## 8. Model metrics joined table


In [ ]:
joined = external.merge(cv, on=['model_name','group','feature_block'], how='left', suffixes=('_external','_cv'))
keep = [
    'model_name','group','feature_block',
    'accuracy','balanced_accuracy','sensitivity','specificity','f1','roc_auc','pr_auc','mcc',
    'cv_accuracy','cv_balanced_accuracy','cv_sensitivity','cv_specificity','cv_f1','cv_roc_auc','cv_pr_auc','cv_mcc'
]
joined[[c for c in keep if c in joined.columns]].sort_values(
    ['accuracy','balanced_accuracy'], ascending=False
).rename(columns=external_metric_names).reset_index(drop=True)
